**Prototype**  

This notebook builds a prototype pypsa network.  
For this, exemplary datasets were generated, post-processing visualizations will be found in *proto_visu.ipynb*.  

In [ ]:
%reset -f

In [ ]:
import pypsa
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
import src.network as network
import src.data as data

In [ ]:
data_path = os.path.join(os.getcwd(), 'data')

In [ ]:
#Importing network data from excel files; consider moving to csv's

buses_df = pd.read_excel(os.path.join(data_path, 'buses.xlsx'), index_col = 0)
buses_df = buses_df.drop(['MV_bus_2'] ) #dropping the second MV bus, will add back when dealing with switching behaviour
#spreading the locations out
buses_df.loc[:, 'x'] *= 10
buses_df.loc[:, 'y'] *= 10

gens_df = pd.read_excel(os.path.join(data_path, 'gens.xlsx'), index_col = 0)
gens_pu_df = pd.read_excel(os.path.join(data_path, 'gen_profiles_pu.xlsx'), index_col = 0)

links_df = pd.read_excel(os.path.join(data_path, 'links.xlsx'), index_col = 0)

loads_df = pd.read_excel(os.path.join(data_path, 'loads.xlsx'), index_col = 0)
loads_pu_df = pd.read_excel(os.path.join(data_path, 'load_profiles_pu.xlsx'), index_col = 0)


In [ ]:
# Adding thermal buses at same locations as electrical buses, to represent thermal energy flows.

LT_df = buses_df.copy().rename(index = lambda x: x + '_LT')
HT_df = buses_df.copy().rename(index = lambda x: x + '_HT')
thermal_buses_df = pd.concat([LT_df, HT_df])
thermal_buses_df['carrier'] = 'thermal'
thermal_buses_df = thermal_buses_df.drop(columns = ['v_nom'])
# thermal_buses_df = thermal_buses_df.drop(index = ['Extern_grid_thermal'])
thermal_buses_df

Pypsa resolves carriers by carrier of buses, carriers on gens and loads, are just for grouping and statistics, and don't mean much.  
What matters is what is connected to which bus!!

In [ ]:
# adding gas buses at same locations as electrical buses, to represent gas energy flows.
gas_buses_df = buses_df.copy(deep = True)
 
gas_buses_df.index = gas_buses_df.index + '_gas'
gas_buses_df.index = gas_buses_df.index.str.replace('Extern_grid_gas', 'Gas_substation')
gas_buses_df.loc['Gas_substation', 'x'] = gas_buses_df.loc['Gas_substation', 'x'] - 0.01 #jsut offsetting it a bit, might be usefule later while visualizing
gas_buses_df['carrier'] = 'gas'
gas_buses_df = gas_buses_df.drop(columns = ['v_nom'])
gas_buses_df

In [ ]:
n = pypsa.Network()

# n.set_snapshots(pd.date_range('2026-01-01 10:00:00', '2026-01-01 12:00:00', freq = 'h'))
n.set_snapshots(pd.date_range('2026-01-01 00:00:00', '2026-12-31 23:00:00', freq = 'h'))
## verify index of load/gen profile input datasets
loads_pu_df = data.clean_index(n.snapshots, loads_pu_df)
gens_pu_df = data.clean_index(n.snapshots, gens_pu_df)

# adding buses
n.add('Bus', buses_df.index, v_nom = buses_df['v_nom'], x = buses_df['x'], y = buses_df['y'], )
n.add('Bus', thermal_buses_df.index, x = thermal_buses_df['x'], y = thermal_buses_df['y'], carrier = thermal_buses_df['carrier'])
n.add('Bus', gas_buses_df.index, x = gas_buses_df['x'], y = gas_buses_df['y'], carrier = gas_buses_df['carrier'])

#adding gens

n.add('Generator', gens_df.index, bus = gens_df['bus'], p_nom = gens_df['p_nom'], marginal_cost = gens_df['marginal_cost'], control = gens_df['control'], carrier = gens_df['carrier'])
n.generators_t.p_max_pu = gens_pu_df[['Roof_pv']]

#adding loads
loads_pset = loads_pu_df.multiply(loads_df['p_max'], axis = 1)
n.add('Load', loads_df.index, bus = loads_df['bus'], p_set = loads_pset, carrier = loads_df['carrier'])

#adding links
# links best represent abstracted DC flows, we don't want AC power dynamics, voltage angles and so on
n.add('Link', links_df.index, bus0 = links_df['bus0'], bus1 = links_df['bus1'], bus2 = links_df['bus2'], carrier = links_df['carrier'], p_nom = links_df['p_nom'], efficiency = links_df['efficiency'], efficiency2 = links_df['efficiency2'], p_min_pu = links_df['p_min_pu'])
# n.links_t.efficiency['Residential_HeatPump'] = gens_pu_df['HP_COP'] #efficiency of heat pump is time dependent
n = network.setCOP_heatpump(n, gens_pu_df) #setting the COP of heat pumps in the network

# adding slack gens in each bus to represent load shedding
for bus in n.buses.index:
    n.add('Generator', f'Slack_{bus}', bus = bus, p_nom = 100, marginal_cost = 10000, control = 'Slack', carrier = 'Load_shed') #carrier helps to group later

# adding a sink to represent export of excess energy to grid
# n.add('Store', 'Export_sink', bus = 'Extern_grid', e_nom = 10000, e_initial = 0, marginal_cost = -10, carrier = 'Export')
n.add('Generator', 'grid_export', bus = 'Extern_grid', p_nom = 1e5, sign = -1, marginal_cost = -15, efficiency = 1, carrier = 'Export')
#adding links between high temp and low temp buses
for bus in buses_df.index:
    if f'{bus}_HT' in n.buses.index and f'{bus}_LT' in n.buses.index:
        
        n.add('Link', f'{bus}_HT_LT', bus0 = f'{bus}_HT', bus1 = f'{bus}_LT', p_nom = 100, efficiency = 0.9, carrier = 'HT_LT_link')
    else:
        print(f"Bus {bus} does not have both HT and LT buses, skipping link creation.")


**CHP, and grid exports**  
The store gives surplus electricity somewhere to flow.  
The surplus could also be re-distributed within the grid, would depend on the costs set.  
Exact flow tracing is not essential for this study, that trouble could be neglected.  
Exports could still be considered, and appropriate export and import tariffs could be used.  


In [ ]:
def register_carriers(n):
    carriers = pd.unique(pd.concat([
        n.buses["carrier"],
        n.generators["carrier"],
        n.links["carrier"],
        n.loads["carrier"],
        n.stores["carrier"]
    ]).dropna())

    colors = {
    "AC":         "#1F77B4",  # blue
    "grid":       "#17BECF",  # cyan
    "solar":      "#FFD700",  # gold
    "gas":        "#7E1408",  # dark red
    "heat":       "#FF8C00",  # dark orange
    "GasBoiler":  "#D62728",  # red
    "Load_shed":  "#000000",  # black
    "CHP":        "#F57C0A",  # orange
    "HP":         "#E8FF00",  # yellow
    "Export":     "#FF00F6",  # magenta
    "DHN":        "#BE08F7",  # purple
    "HT_LT_link": "#9C4FB0",  # muted purple
}
    for carrier in carriers:
        if carrier not in n.carriers.index:
            n.add("Carrier", carrier)
        n.carriers.loc[carrier, "color"] = colors.get(carrier, "gray")  # Assign color or default to gray       

register_carriers(n)

In [ ]:
n.sanitize()

In [ ]:
n.optimize(), n.objective

In [ ]:
#exporting to netcdf

n.export_to_netcdf(os.path.join(data_path, 'proto1.nc'))